# UQ Master Notebook â€” Colab Pro (A100)

Regenerates all Uncertainty Quantification experiment results for the thesis.
Generates fresh JSON output files for all metrics and saves back to Google Drive.

**Estimated runtimes on A100:**
- Cell 7  (MC Dropout, 100 graphs Ã— 30 passes): ~1â€“2 hours
- Cell 8  (Deterministic, 100 graphs):           ~5â€“10 min
- Cells 10â€“18 (metrics analysis):               ~5 min total
- Cell 19 (Exp A, 5 seeds Ã— 30 passes Ã— 100 graphs): ~5â€“10 hours
- Cell 20 (Exp B, 5 models deterministic):       ~20â€“40 min

| Flag | Default | Effect |
|------|---------|--------|
| `SKIP_EXP_A` | `True`  | Skip Experiment A (very long) |
| `SKIP_EXP_B` | `False` | Skip Experiment B (5-model ensemble) |

**Cell order:** Run cells top-to-bottom. Cells 7 and 8 generate the NPZ files that all later cells depend on.

In [ ]:
# â”€â”€ Cell 1: Install PyTorch Geometric â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
import subprocess, sys

def run_quiet(cmd):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if r.returncode != 0:
        print('ERROR:', r.stderr[-1500:])
    return r.returncode == 0

import torch
TORCH_VER = torch.__version__.split('+')[0]
CUDA_TAG  = 'cu' + torch.version.cuda.replace('.', '') if torch.cuda.is_available() else 'cpu'
print(f'PyTorch {TORCH_VER}, CUDA tag: {CUDA_TAG}')

print('Installing torch_geometric...')
run_quiet('pip install torch_geometric --quiet')

print('Installing optional sparse kernels (may fail on some Colab builds â€” OK to ignore)...')
ok = run_quiet(
    f'pip install pyg_lib torch_scatter torch_sparse torch_cluster torch_spline_conv '
    f'-f https://data.pyg.org/whl/torch-{TORCH_VER}+{CUDA_TAG}.html --quiet'
)
if not ok:
    print('  Sparse kernels unavailable â€” continuing with pure-Python fallback (slower but correct).')

import torch_geometric
print(f'torch_geometric {torch_geometric.__version__} ready.')

In [ ]:
# â”€â”€ Cell 2: Mount Google Drive â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
from google.colab import drive
drive.mount('/content/drive')
print('Drive mounted.')

In [ ]:
# â”€â”€ Cell 3: Config â€” paths, model mappings, skip flags â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
import os, json, shutil, gc
import numpy as np
import torch

# â”€â”€ Drive paths â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
DRIVE_BENCHMARKS = '/content/drive/MyDrive/data/TR-C_Benchmarks'
DRIVE_T8_FOLDER  = os.path.join(DRIVE_BENCHMARKS,
                                 'point_net_transf_gat_8th_trial_lower_dropout')
DRIVE_TRAIN_DATA = '/content/drive/MyDrive/data/train_data/dist_not_connected_10k_1pct'

# â”€â”€ Local SSD paths (fast I/O during inference) â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
LOCAL_MODELS  = '/content/models'
LOCAL_RESULTS = '/content/results'
LOCAL_PHASE3  = '/content/results/phase3_results'
LOCAL_CKPT    = '/content/results/checkpoints_mc30'

for d in [LOCAL_MODELS, LOCAL_RESULTS, LOCAL_PHASE3, LOCAL_CKPT]:
    os.makedirs(d, exist_ok=True)

# â”€â”€ Model configs â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
MODEL_FOLDERS = {
    2: 'point_net_transf_gat_2nd_try',
    5: 'point_net_transf_gat_5th_try',
    6: 'point_net_transf_gat_6th_trial_lower_lr',
    7: 'point_net_transf_gat_7th_trial_80_10_10_split',
    8: 'point_net_transf_gat_8th_trial_lower_dropout',
}
DROPOUT_MAP      = {2: 0.3, 5: 0.3, 6: 0.3, 7: 0.3, 8: 0.2}
MODEL_WEIGHTS_R2 = {2: 0.5117, 5: 0.5553, 6: 0.5223, 7: 0.5471, 8: 0.5957}

T8_WEIGHTS_LOCAL = os.path.join(LOCAL_MODELS, 't8', 'model.pth')

# â”€â”€ Experiment skip flags â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
# Set SKIP_EXP_A = False to re-run Exp A (5 seeds, ~5-10 hours)
SKIP_EXP_A = True
# Set SKIP_EXP_B = True to skip Exp B (5-model ensemble, ~20-40 min)
SKIP_EXP_B = False

# â”€â”€ Device â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
if DEVICE.type == 'cuda':
    print(f'GPU:   {torch.cuda.get_device_name(0)}')
    print(f'VRAM:  {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
print()
print(f'SKIP_EXP_A={SKIP_EXP_A}  SKIP_EXP_B={SKIP_EXP_B}')

In [ ]:
# â”€â”€ Cell 4: Copy model weights from Drive â†’ local SSD â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
def copy_model(trial_num, verbose=True):
    """Copy trained_model/model.pth from Drive to /content/models/tN/model.pth."""
    folder = MODEL_FOLDERS[trial_num]
    src    = os.path.join(DRIVE_BENCHMARKS, folder, 'trained_model', 'model.pth')
    dst_dir = os.path.join(LOCAL_MODELS, f't{trial_num}')
    dst    = os.path.join(dst_dir, 'model.pth')

    if os.path.exists(dst):
        if verbose: print(f'  T{trial_num}: already cached locally.')
        return dst
    if not os.path.exists(src):
        print(f'  T{trial_num}: WARNING â€” not found at {src}')
        return None

    os.makedirs(dst_dir, exist_ok=True)
    shutil.copy2(src, dst)
    size_mb = os.path.getsize(dst) / 1e6
    if verbose: print(f'  T{trial_num}: copied {size_mb:.1f} MB.')
    return dst

print('Copying model weights from Drive...')
for trial in [8, 2, 5, 6, 7]:   # T8 first; others needed for Exp B
    copy_model(trial)

assert os.path.exists(T8_WEIGHTS_LOCAL), \
    f'T8 model not found at {T8_WEIGHTS_LOCAL}. Check Drive path.'
print('\nAll models present. T8 verified.')

In [ ]:
# â”€â”€ Cell 5: Inline model definition â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
# Stripped version: no wandb, no ABC, no train_model()
# Source: code/scripts/gnn/models/base_gnn.py + point_net_transf_gat.py
# â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
import torch
import torch.nn as nn
import torch.nn.init as init
import numpy as np
from torch_geometric.nn import (
    Sequential as GeoSequential,
    TransformerConv,
    GATConv,
    PointNetConv,
)


class BaseGNN(nn.Module):
    """Base GNN class (stripped: no ABC, no wandb, no train_model)."""
    def __init__(self, in_channels, out_channels, dropout=0.3,
                 use_dropout=False, predict_mode_stats=False,
                 dtype=torch.float32, log_to_wandb=False):
        super().__init__()
        self.in_channels        = in_channels
        self.out_channels       = out_channels
        self.dropout            = dropout
        self.use_dropout        = use_dropout
        self.predict_mode_stats = predict_mode_stats
        self.dtype              = dtype
        self.log_to_wandb       = log_to_wandb

    def define_layers(self): pass
    def forward(self, data): pass

    def initialize_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)


class PointNetTransfGAT(BaseGNN):
    def __init__(self,
                 in_channels=5, out_channels=1,
                 point_net_conv_layer_structure_local_mlp=None,
                 point_net_conv_layer_structure_global_mlp=None,
                 gat_conv_layer_structure=None,
                 dropout=0.3, use_dropout=False,
                 predict_mode_stats=False,
                 dtype=torch.float32, log_to_wandb=False):
        if point_net_conv_layer_structure_local_mlp is None:
            point_net_conv_layer_structure_local_mlp = [256]
        if point_net_conv_layer_structure_global_mlp is None:
            point_net_conv_layer_structure_global_mlp = [512]
        if gat_conv_layer_structure is None:
            gat_conv_layer_structure = [128, 256, 512]
        super().__init__(
            in_channels=in_channels, out_channels=out_channels,
            dropout=dropout, use_dropout=use_dropout,
            predict_mode_stats=predict_mode_stats,
            dtype=dtype, log_to_wandb=log_to_wandb)
        self.pnc_local  = point_net_conv_layer_structure_local_mlp
        self.pnc_global = point_net_conv_layer_structure_global_mlp
        self.gat_conv   = gat_conv_layer_structure
        self.define_layers()
        self.initialize_weights()

    def define_layers(self):
        if self.use_dropout:
            self.dropout_layer = nn.Dropout(self.dropout)
        self.point_net_conv_1 = self._create_pnc(
            self.gat_conv[0], is_first=True, is_last=False)
        self.point_net_conv_2 = self._create_pnc(
            self.gat_conv[0], is_first=False, is_last=True)
        layers = self._define_gat_layers()
        self.gat_graph_layers = GeoSequential('x, edge_index', layers)
        self.gat_final        = GATConv(64, 1)

    def forward(self, data):
        x          = data.x.to(self.dtype)
        edge_index = data.edge_index
        pos1 = data.pos[:, 0, :]
        pos2 = data.pos[:, 1, :]
        x = self.point_net_conv_1(x, pos1, edge_index)
        x = self.point_net_conv_2(x, pos2, edge_index)
        x = self.gat_graph_layers(x, edge_index)
        return self.gat_final(x, edge_index)

    def _define_gat_layers(self):
        layers = []
        for i in range(len(self.gat_conv) - 1):
            layers.append((
                TransformerConv(self.gat_conv[i],
                                int(self.gat_conv[i + 1] / 4), heads=4),
                'x, edge_index -> x'))
            layers.append(nn.ReLU(inplace=True))
            if self.use_dropout:
                layers.append(self.dropout_layer)
        layers.append((GATConv(self.gat_conv[-1], 64), 'x, edge_index -> x'))
        return layers

    def _create_pnc(self, gat_start, is_first=False, is_last=False):
        offset = 2  # 2D pos appended to features
        local_layers = []
        in_dim = (self.in_channels + offset) if is_first else (self.pnc_global[-1] + offset)
        local_layers.append(nn.Linear(in_dim, self.pnc_local[0]))
        local_layers.append(nn.ReLU())
        if self.use_dropout: local_layers.append(self.dropout_layer)
        for i in range(len(self.pnc_local) - 1):
            local_layers.append(nn.Linear(self.pnc_local[i], self.pnc_local[i + 1]))
            local_layers.append(nn.ReLU())
            if self.use_dropout: local_layers.append(self.dropout_layer)
        local_MLP = nn.Sequential(*local_layers)

        global_layers = []
        global_layers.append(nn.Linear(self.pnc_local[-1], self.pnc_global[0]))
        global_layers.append(nn.ReLU())
        if self.use_dropout: global_layers.append(self.dropout_layer)
        for i in range(len(self.pnc_global) - 1):
            global_layers.append(nn.Linear(self.pnc_global[i], self.pnc_global[i + 1]))
            global_layers.append(nn.ReLU())
            if self.use_dropout: global_layers.append(self.dropout_layer)
        out_dim = gat_start if is_last else self.pnc_global[-1]
        global_layers.append(nn.Linear(self.pnc_global[-1], out_dim))
        global_layers.append(nn.ReLU())
        if self.use_dropout: global_layers.append(self.dropout_layer)
        global_MLP = nn.Sequential(*global_layers)
        return PointNetConv(local_nn=local_MLP, global_nn=global_MLP)

    def initialize_weights(self):
        super().initialize_weights()
        for m in self.modules():
            if isinstance(m, PointNetConv):
                for name, param in list(m.local_nn.named_parameters()) + list(m.global_nn.named_parameters()):
                    if param.dim() > 1:
                        init.kaiming_normal_(param, mode='fan_out', nonlinearity='relu')
                    else:
                        init.zeros_(param)


# â”€â”€ Helper functions â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

def load_model(weights_path, dropout=0.2, device=None):
    """
    Load PointNetTransfGAT with GATConv weight remapping.
    T2-T8 models saved with old PyG (lin_src/lin_dst).
    PyG 2.3+ consolidates to a single lin.weight.
    """
    if device is None:
        device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = PointNetTransfGAT(
        in_channels=5, out_channels=1,
        point_net_conv_layer_structure_local_mlp=[256],
        point_net_conv_layer_structure_global_mlp=[512],
        gat_conv_layer_structure=[128, 256, 512],
        dropout=dropout, use_dropout=True,
        predict_mode_stats=False, log_to_wandb=False,
    )
    raw = torch.load(weights_path, map_location=device, weights_only=False)
    remapped = {}
    for k, v in raw.items():
        if '.lin_src.weight' in k:
            # Old PyG (lin_src/lin_dst) -> PyG 2.3+ uses single lin
            remapped[k.replace('.lin_src.weight', '.lin.weight')] = v
        elif '.lin_dst.weight' in k:
            pass  # drop: lin_src already mapped to lin (weights are tied)
        else:
            remapped[k] = v
    model.load_state_dict(remapped, strict=True)
    return model.to(device)


def mc_infer(model, data, S=30, device=None):
    """
    MC Dropout: S forward passes with dropout ON, BatchNorm frozen.
    Returns (mean, std) where std uses ddof=0 (biased estimator).
    """
    if device is None:
        device = next(model.parameters()).device
    model.train()
    for m in model.modules():
        if isinstance(m, torch.nn.modules.batchnorm._BatchNorm):
            m.eval()
    preds = []
    with torch.no_grad():
        for _ in range(S):
            out = model(data.to(device))
            if isinstance(out, tuple):
                out = out[0]
            preds.append(out.squeeze().cpu().numpy())
    preds = np.array(preds)   # (S, N_nodes)
    return preds.mean(axis=0), preds.std(axis=0)  # ddof=0 matches original scripts


def det_infer(model, data, device=None):
    """Deterministic: model.eval(), single forward pass."""
    if device is None:
        device = next(model.parameters()).device
    model.eval()
    with torch.no_grad():
        out = model(data.to(device))
        if isinstance(out, tuple):
            out = out[0]
    return out.squeeze().cpu().numpy()


# Quick sanity: instantiate model and count params
_tmp = PointNetTransfGAT(
    in_channels=5, out_channels=1,
    point_net_conv_layer_structure_local_mlp=[256],
    point_net_conv_layer_structure_global_mlp=[512],
    gat_conv_layer_structure=[128, 256, 512],
    dropout=0.2, use_dropout=True)
n_params = sum(p.numel() for p in _tmp.parameters())
print(f'PointNetTransfGAT: {n_params:,} parameters')
del _tmp
print('Model definition and helpers ready.')

In [ ]:
# â”€â”€ Cell 6: Load test data â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
# Tries in order:
#   1. Local cache /content/models/test_dl.pt
#   2. Drive: T8_FOLDER/data_created_during_training/test_dl.pt
#   3. Reconstruct from datalist_batch_*.pt (80/10/10 split, last 10% = test)

LOCAL_TEST_DL = os.path.join(LOCAL_MODELS, 'test_dl.pt')
DRIVE_TEST_DL = os.path.join(DRIVE_T8_FOLDER,
                              'data_created_during_training', 'test_dl.pt')

if os.path.exists(LOCAL_TEST_DL):
    print(f'Loading test data from local cache: {LOCAL_TEST_DL}')
    TEST_DATA = torch.load(LOCAL_TEST_DL, weights_only=False)

elif os.path.exists(DRIVE_TEST_DL):
    print(f'Loading test data from Drive: {DRIVE_TEST_DL}')
    TEST_DATA = torch.load(DRIVE_TEST_DL, weights_only=False)
    torch.save(TEST_DATA, LOCAL_TEST_DL)
    print(f'Cached locally: {LOCAL_TEST_DL}')

else:
    print('test_dl.pt not found â€” reconstructing from datalist batches...')
    all_graphs = []
    for i in range(1, 21):
        bp = os.path.join(DRIVE_TRAIN_DATA, f'datalist_batch_{i}.pt')
        if not os.path.exists(bp):
            raise FileNotFoundError(f'Batch file missing: {bp}')
        batch = torch.load(bp, weights_only=False)
        all_graphs.extend(batch)
        print(f'  Batch {i:02d}/20: {len(batch)} graphs (total {len(all_graphs)})')

    n_total    = len(all_graphs)
    test_start = int(0.9 * n_total)   # 80/10/10 -> test = last 10%
    TEST_DATA  = all_graphs[test_start:]
    print(f'Reconstructed: {len(TEST_DATA)} test graphs (indices {test_start}-{n_total-1})')
    torch.save(TEST_DATA, LOCAL_TEST_DL)
    print(f'Saved to: {LOCAL_TEST_DL}')

N_GRAPHS = len(TEST_DATA)
sample   = TEST_DATA[0]
TOTAL_NODES = sum(d.x.shape[0] for d in TEST_DATA)
NODES_PER_GRAPH = TEST_DATA[0].x.shape[0]

print(f'\nTest set: {N_GRAPHS} graphs, {TOTAL_NODES:,} total nodes')
print(f'  Nodes per graph (sample): {NODES_PER_GRAPH:,}')
print(f'  x shape:          {sample.x.shape}')
print(f'  pos shape:        {sample.pos.shape}')
print(f'  edge_index shape: {sample.edge_index.shape}')
print(f'  y shape:          {sample.y.shape}')

In [ ]:
# â”€â”€ Cell 7: MC Dropout T8, S=30, 100 graphs â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
# Per-graph checkpointing: resume from any interruption.
# Output: /content/results/mc_dropout_full_100graphs_mc30.npz
# Expected: R2~0.5957, Spearman rho~0.464
import time
from tqdm.auto import tqdm
from scipy.stats import spearmanr

MC_NPZ_OUT = os.path.join(LOCAL_RESULTS, 'mc_dropout_full_100graphs_mc30.npz')
S_MC       = 30

print(f'Loading T8 for MC Dropout (S={S_MC})...')
model_mc = load_model(T8_WEIGHTS_LOCAL, dropout=DROPOUT_MAP[8], device=DEVICE)
print('T8 loaded.')

all_preds, all_uncs, all_targets = [], [], []
t_start = time.time()

for g_idx in tqdm(range(N_GRAPHS), desc='MC Dropout T8'):
    ckpt = os.path.join(LOCAL_CKPT, f'graph_{g_idx:04d}.npz')

    if os.path.exists(ckpt):
        d     = np.load(ckpt)
        mu    = d['predictions']
        sigma = d['uncertainties']
        y     = d['targets']
    else:
        data  = TEST_DATA[g_idx]
        mu, sigma = mc_infer(model_mc, data, S=S_MC, device=DEVICE)
        y     = data.y.squeeze().cpu().numpy()
        np.savez_compressed(ckpt,
                            predictions=mu.astype(np.float32),
                            uncertainties=sigma.astype(np.float32),
                            targets=y.astype(np.float32))

    all_preds.append(mu)
    all_uncs.append(sigma)
    all_targets.append(y)

    if (g_idx + 1) % 10 == 0:
        elapsed   = time.time() - t_start
        remaining = elapsed / (g_idx + 1) * (N_GRAPHS - g_idx - 1)
        tqdm.write(f'  {g_idx+1}/{N_GRAPHS} | {elapsed/60:.1f}min elapsed | ~{remaining/60:.1f}min left')

all_preds   = np.concatenate(all_preds).astype(np.float32)
all_uncs    = np.concatenate(all_uncs).astype(np.float32)
all_targets = np.concatenate(all_targets).astype(np.float32)

np.savez_compressed(MC_NPZ_OUT,
                    predictions=all_preds,
                    uncertainties=all_uncs,
                    targets=all_targets)

r2_mc  = 1 - np.sum((all_targets - all_preds)**2) / np.sum((all_targets - all_targets.mean())**2)
mae_mc = np.mean(np.abs(all_targets - all_preds))
rho_mc, _ = spearmanr(all_uncs, np.abs(all_targets - all_preds))

print(f'\nMC Dropout NPZ saved: {MC_NPZ_OUT}')
print(f'  Nodes:         {len(all_preds):,}')
print(f'  R2:            {r2_mc:.4f}  (expected ~0.5957)')
print(f'  MAE:           {mae_mc:.4f}')
print(f'  Spearman rho:  {rho_mc:.4f}  (expected ~0.464)')
print(f'  Unc mean/std:  {all_uncs.mean():.4f} / {all_uncs.std():.4f}')
print(f'  Total time:    {(time.time()-t_start)/60:.1f} min')

del model_mc
gc.collect()
if DEVICE.type == 'cuda':
    torch.cuda.empty_cache()

In [ ]:
# â”€â”€ Cell 8: Deterministic T8, 100 graphs â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
# Output: /content/results/deterministic_full_100graphs.npz
import time
from tqdm.auto import tqdm

DET_NPZ_OUT = os.path.join(LOCAL_RESULTS, 'deterministic_full_100graphs.npz')

print('Loading T8 for deterministic inference...')
model_det = load_model(T8_WEIGHTS_LOCAL, dropout=DROPOUT_MAP[8], device=DEVICE)
model_det.eval()
print('T8 loaded.')

det_preds, det_targets = [], []
t_det = time.time()

for data in tqdm(TEST_DATA, desc='Deterministic T8'):
    pred = det_infer(model_det, data, device=DEVICE)
    y    = data.y.squeeze().cpu().numpy()
    det_preds.append(pred)
    det_targets.append(y)

det_preds   = np.concatenate(det_preds).astype(np.float32)
det_targets = np.concatenate(det_targets).astype(np.float32)

np.savez_compressed(DET_NPZ_OUT, predictions=det_preds, targets=det_targets)

r2_det  = 1 - np.sum((det_targets - det_preds)**2) / np.sum((det_targets - det_targets.mean())**2)
mae_det = np.mean(np.abs(det_targets - det_preds))

print(f'\nDeterministic NPZ saved: {DET_NPZ_OUT}')
print(f'  Nodes: {len(det_preds):,}')
print(f'  R2:    {r2_det:.4f}  (expected ~0.5957)')
print(f'  MAE:   {mae_det:.4f}')
print(f'  Time:  {(time.time()-t_det)/60:.1f} min')

del model_det
gc.collect()
if DEVICE.type == 'cuda':
    torch.cuda.empty_cache()

In [ ]:
# â”€â”€ Cell 9: Sync NPZ files back to Drive â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
DRIVE_UQ = os.path.join(DRIVE_T8_FOLDER, 'uq_results')
os.makedirs(DRIVE_UQ, exist_ok=True)

for fname in ['mc_dropout_full_100graphs_mc30.npz', 'deterministic_full_100graphs.npz']:
    src = os.path.join(LOCAL_RESULTS, fname)
    dst = os.path.join(DRIVE_UQ, fname)
    if os.path.exists(src):
        shutil.copy2(src, dst)
        print(f'  Synced: {fname}  ({os.path.getsize(dst)/1e6:.1f} MB)')
    else:
        print(f'  SKIP (not found locally): {fname}')

print('Drive sync done.')

In [ ]:
# â”€â”€ Cell 10: CRPS (Continuous Ranked Probability Score) â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
# Gaussian closed-form: sigma*(z*(2*Phi(z)-1) + 2*phi(z) - 1/sqrt(pi))
# Expected: CRPS~3.38, CRPS/MAE~0.857
from scipy.stats import norm as _norm

def compute_crps_gaussian(mu, sigma, y):
    z    = (y - mu) / (sigma + 1e-12)
    crps = sigma * (z * (2 * _norm.cdf(z) - 1) + 2 * _norm.pdf(z) - 1.0 / np.sqrt(np.pi))
    return crps

npz   = np.load(os.path.join(LOCAL_RESULTS, 'mc_dropout_full_100graphs_mc30.npz'))
mu    = npz['predictions'].astype(np.float64)
sigma = npz['uncertainties'].astype(np.float64)
y     = npz['targets'].astype(np.float64)

crps_vals  = compute_crps_gaussian(mu, sigma, y)
mean_crps  = float(np.mean(crps_vals))
mae        = float(np.mean(np.abs(y - mu)))
crps_ratio = mean_crps / mae

results_crps = {
    'mean_crps':     mean_crps,
    'mae':           mae,
    'crps_mae_ratio': crps_ratio,
    'crps_std':      float(np.std(crps_vals)),
    'n_nodes':       int(len(y)),
}

print(f'CRPS mean:  {mean_crps:.4f}  (expected ~3.38)')
print(f'CRPS/MAE:   {crps_ratio:.4f}  (expected ~0.857)')
print(f'MAE:        {mae:.4f}')

with open(os.path.join(LOCAL_PHASE3, 'crps_t8.json'), 'w') as f:
    json.dump(results_crps, f, indent=2)
print('Saved: crps_t8.json')

In [ ]:
# â”€â”€ Cell 11: PIT (Probability Integral Transform) â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
# Expected: mean~0.433, std~0.399, first-bin~28.4%, KS~0.245
from scipy.stats import norm as _norm, kstest

npz   = np.load(os.path.join(LOCAL_RESULTS, 'mc_dropout_full_100graphs_mc30.npz'))
mu    = npz['predictions'].astype(np.float64)
sigma = npz['uncertainties'].astype(np.float64)
y     = npz['targets'].astype(np.float64)

pit_vals = _norm.cdf((y - mu) / (sigma + 1e-12))

# KS test on 50K subsample (fixed seed=42 per original script)
rng     = np.random.default_rng(42)
sub_idx = rng.choice(len(pit_vals), size=50000, replace=False)
ks_stat, ks_p = kstest(pit_vals[sub_idx], 'uniform')

counts, _ = np.histogram(pit_vals, bins=20, range=(0, 1))
first_bin_pct = float(counts[0] / len(pit_vals) * 100)

results_pit = {
    'pit_mean':      float(np.mean(pit_vals)),
    'pit_std':       float(np.std(pit_vals)),
    'first_bin_pct': first_bin_pct,
    'ks_statistic':  float(ks_stat),
    'ks_p_value':    float(ks_p),
    'n_bins':        20,
    'bin_counts':    counts.tolist(),
    'n_nodes':       int(len(y)),
}

print(f'PIT mean:      {results_pit["pit_mean"]:.3f}  (expected ~0.433)')
print(f'PIT std:       {results_pit["pit_std"]:.3f}  (expected ~0.399)')
print(f'First bin %:   {first_bin_pct:.1f}%  (expected ~28.4%)')
print(f'KS stat:       {ks_stat:.3f}  (expected ~0.245)')
print(f'KS p-value:    {ks_p:.2e}')

with open(os.path.join(LOCAL_PHASE3, 'pit_t8.json'), 'w') as f:
    json.dump(results_pit, f, indent=2)
print('Saved: pit_t8.json')

In [ ]:
# â”€â”€ Cell 12: Temperature Scaling Calibration â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
# Split: 30% val / 70% test (np.random.seed(42), node-level)
# Grid search T on val: logspace(-1,2,50) then minimize_scalar fine-tune
# ECE: percentile-binning by sigma, check |error| < sigma -> 68.27% expected
# Expected: T~2.7025, ECE_full~0.265, ECE_test_before~0.269, ECE_test_after~0.048, improvement~82%
from scipy.optimize import minimize_scalar

def compute_ece(mu_, sigma_, y_, n_bins=10):
    sort_idx = np.argsort(sigma_)
    s, m, t  = sigma_[sort_idx], mu_[sort_idx], y_[sort_idx]
    n, bsz   = len(s), len(s) // n_bins
    ece_bins = []
    for b in range(n_bins):
        lo = b * bsz
        hi = lo + bsz if b < n_bins - 1 else n
        within = np.abs(t[lo:hi] - m[lo:hi]) < s[lo:hi]
        ece_bins.append(abs(within.mean() - 0.6827))
    return float(np.mean(ece_bins))

npz   = np.load(os.path.join(LOCAL_RESULTS, 'mc_dropout_full_100graphs_mc30.npz'))
mu    = npz['predictions'].astype(np.float64)
sigma = npz['uncertainties'].astype(np.float64)
y     = npz['targets'].astype(np.float64)

np.random.seed(42)
idx     = np.random.permutation(len(mu))
val_end = int(0.30 * len(mu))
vi, ti  = idx[:val_end], idx[val_end:]
mu_v, sigma_v, y_v = mu[vi], sigma[vi], y[vi]
mu_t, sigma_t, y_t = mu[ti], sigma[ti], y[ti]

T_grid   = np.logspace(-1, 2, 50)
ece_grid = [compute_ece(mu_v, sigma_v * T, y_v) for T in T_grid]
T_init   = T_grid[np.argmin(ece_grid)]

res    = minimize_scalar(lambda T: compute_ece(mu_v, sigma_v * T, y_v),
                         bounds=(T_init / 2, T_init * 2), method='bounded')
T_opt  = float(res.x)

ECE_full   = compute_ece(mu, sigma, y)
ECE_before = compute_ece(mu_t, sigma_t, y_t)
ECE_after  = compute_ece(mu_t, sigma_t * T_opt, y_t)
impr_pct   = (ECE_before - ECE_after) / ECE_before * 100

results_ts = {
    'T_opt':            T_opt,
    'ECE_full':         ECE_full,
    'ECE_test_before':  ECE_before,
    'ECE_test_after':   ECE_after,
    'improvement_pct':  impr_pct,
}

print(f'T_opt:              {T_opt:.4f}  (expected ~2.7025)')
print(f'ECE full:           {ECE_full:.3f}  (expected ~0.265)')
print(f'ECE test before:    {ECE_before:.3f}  (expected ~0.269)')
print(f'ECE test after:     {ECE_after:.3f}  (expected ~0.048)')
print(f'Improvement:        {impr_pct:.1f}%  (expected ~82%)')

with open(os.path.join(LOCAL_PHASE3, 'temperature_scaling_t8.json'), 'w') as f:
    json.dump(results_ts, f, indent=2)
print('Saved: temperature_scaling_t8.json')

In [ ]:
# â”€â”€ Cell 13: Selective Prediction â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
# Sort ALL nodes by sigma ascending (least uncertain first).
# At each retention level r, keep bottom r*N nodes and compute MAE.
# Expected reductions: -18.3% / -41.2% / -54.6% / -73.3% at 90%/50%/25%/10%

RETENTION_LEVELS = [1.0, 0.95, 0.90, 0.85, 0.80, 0.75, 0.70, 0.60, 0.50, 0.40, 0.30, 0.25, 0.10]

npz   = np.load(os.path.join(LOCAL_RESULTS, 'mc_dropout_full_100graphs_mc30.npz'))
mu    = npz['predictions'].astype(np.float64)
sigma = npz['uncertainties'].astype(np.float64)
y     = npz['targets'].astype(np.float64)

sort_idx = np.argsort(sigma)      # ascending: index 0 = least uncertain
full_mae = float(np.mean(np.abs(y - mu)))

ret_results = []
for r in RETENTION_LEVELS:
    k       = max(1, int(r * len(mu)))
    kept    = sort_idx[:k]
    mae_r   = float(np.mean(np.abs(y[kept] - mu[kept])))
    reduc   = (mae_r - full_mae) / full_mae * 100
    ret_results.append({'retention': r, 'k': k, 'mae': mae_r, 'mae_reduction_pct': reduc})

results_sel = {'full_mae': full_mae, 'retention_results': ret_results}

print(f'  {"Retention":>10} {"k":>9} {"MAE":>10} {"Reduction%":>12}')
print('  ' + '-' * 45)
for e in ret_results:
    print(f'  {e["retention"]*100:5.0f}%    {e["k"]:8,}  {e["mae"]:9.4f}  {e["mae_reduction_pct"]:10.1f}%')

print('\nExpected: -18.3% / -41.2% / -54.6% / -73.3% at 90% / 50% / 25% / 10%')

with open(os.path.join(LOCAL_PHASE3, 'selective_prediction_s30.json'), 'w') as f:
    json.dump(results_sel, f, indent=2)
print('Saved: selective_prediction_s30.json')

In [ ]:
# â”€â”€ Cell 14: NLL (Negative Log-Likelihood) â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
# Formula: 0.5 * mean[ log(2*pi*sigma^2) + (y-mu)^2 / sigma^2 ]
# Expected: NLL raw ~21.65; temp-scaled NLL uses T_opt from Cell 12

with open(os.path.join(LOCAL_PHASE3, 'temperature_scaling_t8.json')) as f:
    ts_data = json.load(f)
T_opt = ts_data.get('T_opt', 2.7025)

npz   = np.load(os.path.join(LOCAL_RESULTS, 'mc_dropout_full_100graphs_mc30.npz'))
mu    = npz['predictions'].astype(np.float64)
sigma = npz['uncertainties'].astype(np.float64)
y     = npz['targets'].astype(np.float64)

def compute_nll(mu_, sigma_, y_):
    return float(0.5 * np.mean(
        np.log(2 * np.pi * sigma_**2) + (y_ - mu_)**2 / sigma_**2
    ))

nll_raw    = compute_nll(mu, sigma, y)
nll_scaled = compute_nll(mu, sigma * T_opt, y)

results_nll = {
    'nll_raw':        nll_raw,
    'T_opt':          T_opt,
    'nll_temp_scaled': nll_scaled,
}

print(f'NLL raw:          {nll_raw:.4f}  (expected ~21.65)')
print(f'NLL temp-scaled:  {nll_scaled:.4f}  (T={T_opt:.4f})')

with open(os.path.join(LOCAL_PHASE3, 'nll_results.json'), 'w') as f:
    json.dump(results_nll, f, indent=2)
print('Saved: nll_results.json')

In [ ]:
# â”€â”€ Cell 15: Bootstrap CI for Spearman rho â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
# Block bootstrap: resample 100 graphs, compute per-graph rho, B=10000 replicates
# Expected: mean~0.464, CI~[0.460, 0.469]
from scipy.stats import spearmanr

npz   = np.load(os.path.join(LOCAL_RESULTS, 'mc_dropout_full_100graphs_mc30.npz'))
mu    = npz['predictions'].astype(np.float64)
sigma = npz['uncertainties'].astype(np.float64)
y     = npz['targets'].astype(np.float64)

N_G          = N_GRAPHS          # 100
# Build per-graph slice boundaries from actual TEST_DATA sizes
# (robust to variable-size graphs; avoids off-by-one if counts differ)
_g_sizes  = [TEST_DATA[g].x.shape[0] for g in range(N_G)]
_g_starts = [int(sum(_g_sizes[:g])) for g in range(N_G)]
NPG = int(np.mean(_g_sizes))   # informational only
B_BOOT       = 10000
np.random.seed(42)

abs_error = np.abs(y - mu)
per_g_rho = []
for g in range(N_G):
    s, e = _g_starts[g], _g_starts[g] + _g_sizes[g]
    rho_g, _ = spearmanr(sigma[s:e], abs_error[s:e])
    per_g_rho.append(float(rho_g))
per_g_rho = np.array(per_g_rho)

boot_means = np.array([
    per_g_rho[np.random.choice(N_G, size=N_G, replace=True)].mean()
    for _ in range(B_BOOT)
])

mean_rho = float(per_g_rho.mean())
ci_lo    = float(np.percentile(boot_means, 2.5))
ci_hi    = float(np.percentile(boot_means, 97.5))

results_boot = {
    'mean_rho':        mean_rho,
    'ci_lo':           ci_lo,
    'ci_hi':           ci_hi,
    'B':               B_BOOT,
    'nodes_per_graph': NPG,
    'n_graphs':        N_G,
    'per_graph_rho':   per_g_rho.tolist(),
}

print(f'Mean Spearman rho:  {mean_rho:.4f}  (expected ~0.464)')
print(f'95% CI:             [{ci_lo:.3f}, {ci_hi:.3f}]  (expected [0.460, 0.469])')

with open(os.path.join(LOCAL_PHASE3, 'bootstrap_ci_results.json'), 'w') as f:
    json.dump(results_boot, f, indent=2)
print('Saved: bootstrap_ci_results.json')

In [ ]:
# â”€â”€ Cell 16: AUROC (uncertainty as error detector) â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
# is_high_error = (|error| > median_error); score = sigma
# Expected: AUROC ~0.755
from sklearn.metrics import roc_auc_score

npz   = np.load(os.path.join(LOCAL_RESULTS, 'mc_dropout_full_100graphs_mc30.npz'))
mu    = npz['predictions'].astype(np.float64)
sigma = npz['uncertainties'].astype(np.float64)
y     = npz['targets'].astype(np.float64)

abs_error     = np.abs(y - mu)
median_error  = np.median(abs_error)
is_high_error = (abs_error > median_error).astype(int)
auroc         = float(roc_auc_score(is_high_error, sigma))

results_auroc = {
    'auroc':                   auroc,
    'median_error_threshold':  float(median_error),
    'n_nodes':                 int(len(y)),
}

print(f'AUROC:                  {auroc:.4f}  (expected ~0.755)')
print(f'Median error threshold: {median_error:.4f}')

with open(os.path.join(LOCAL_PHASE3, 'auroc_results.json'), 'w') as f:
    json.dump(results_auroc, f, indent=2)
print('Saved: auroc_results.json')

In [ ]:
# â”€â”€ Cell 17: Adaptive Conformal Prediction â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
# Split: first 50 graphs = calibration, last 50 = test (matching conformal_from_mc.py)
# Standard: absolute residuals -> global q -> per-graph coverage range
# Adaptive: sigma-scaled residuals -> q_adapt -> per-graph coverage range
# Expected: Standard per-graph [62.9%, 98.6%], Adaptive per-graph [90.0%, 96.2%]

npz    = np.load(os.path.join(LOCAL_RESULTS, 'mc_dropout_full_100graphs_mc30.npz'))
mu_all = npz['predictions'].astype(np.float64)
sg_all = npz['uncertainties'].astype(np.float64)
y_all  = npz['targets'].astype(np.float64)

# Build per-graph slice boundaries from actual TEST_DATA sizes
_g_sizes_all  = [TEST_DATA[g].x.shape[0] for g in range(N_GRAPHS)]
_g_starts_all = [int(sum(_g_sizes_all[:g])) for g in range(N_GRAPHS)]
NPG_C = int(np.mean(_g_sizes_all))   # informational only
alpha  = 0.10
eps    = 1e-6

# calibration = first 50 graphs, test = last 50
cal_end  = int(sum(_g_sizes_all[:50]))
mu_c,  sg_c,  y_c  = mu_all[:cal_end],  sg_all[:cal_end],  y_all[:cal_end]
mu_ts, sg_ts, y_ts = mu_all[cal_end:],  sg_all[cal_end:],  y_all[cal_end:]

def conformal_q(residuals, alpha):
    """Standard conformal quantile: ceil((n+1)*(1-alpha))/n."""
    n       = len(residuals)
    q_level = min(np.ceil((n + 1) * (1 - alpha)) / n, 1.0)
    return float(np.quantile(residuals, q_level, method='higher'))

# â”€â”€ Standard conformal (absolute residuals) â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
r_abs      = np.abs(y_c - mu_c)
q_std      = conformal_q(r_abs, alpha)

std_covs = []
for g in range(50):
    s = _g_starts_all[50 + g] - cal_end
    e = s + _g_sizes_all[50 + g]
    cov  = np.abs(y_ts[s:e] - mu_ts[s:e]) <= q_std
    std_covs.append(float(cov.mean() * 100))

# â”€â”€ Adaptive conformal (sigma-scaled residuals) â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
r_scaled   = np.abs(y_c - mu_c) / (sg_c + eps)
q_adp      = conformal_q(r_scaled, alpha)

adp_covs = []
for g in range(50):
    s = _g_starts_all[50 + g] - cal_end
    e = s + _g_sizes_all[50 + g]
    cov  = np.abs(y_ts[s:e] - mu_ts[s:e]) <= q_adp * (sg_ts[s:e] + eps)
    adp_covs.append(float(cov.mean() * 100))

# â”€â”€ Global coverages (for sanity check; conformal_standard.json has this) â”€â”€â”€â”€â”€
cov_std_global = float(np.mean(np.abs(y_ts - mu_ts) <= q_std) * 100)
cov_adp_global = float(np.mean(np.abs(y_ts - mu_ts) <= q_adp * (sg_ts + eps)) * 100)

results_conf = {
    'alpha':                       alpha,
    'eps':                         eps,
    'q_standard':                  q_std,
    'q_adapt':                     q_adp,
    'standard_global_coverage_pct': cov_std_global,
    'adaptive_global_coverage_pct': cov_adp_global,
    'standard_per_graph_min_pct':  float(np.min(std_covs)),
    'standard_per_graph_max_pct':  float(np.max(std_covs)),
    'standard_per_graph_mean_pct': float(np.mean(std_covs)),
    'adaptive_per_graph_min_pct':  float(np.min(adp_covs)),
    'adaptive_per_graph_max_pct':  float(np.max(adp_covs)),
    'adaptive_per_graph_mean_pct': float(np.mean(adp_covs)),
    'standard_per_graph':          std_covs,
    'adaptive_per_graph':          adp_covs,
}

print(f'q_standard={q_std:.4f},  q_adapt={q_adp:.4f}')
print(f'Standard per-graph coverage: [{np.min(std_covs):.1f}%, {np.max(std_covs):.1f}%]')
print(f'  Expected: [62.9%, 98.6%]')
print(f'Adaptive per-graph coverage: [{np.min(adp_covs):.1f}%, {np.max(adp_covs):.1f}%]')
print(f'  Expected: [90.0%, 96.2%]')
print(f'Global: standard={cov_std_global:.2f}%, adaptive={cov_adp_global:.2f}%')

with open(os.path.join(LOCAL_PHASE3, 'adaptive_conformal_results.json'), 'w') as f:
    json.dump(results_conf, f, indent=2)
print('Saved: adaptive_conformal_results.json')

In [ ]:
# â”€â”€ Cell 18: S-Convergence (10 graphs, S up to 50) â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
# Check how MC uncertainty estimate converges as S increases.
# T8 dropout=0.2  (original script had a 0.3 bug â€” fixed here).
# Per-graph seed: torch.manual_seed(SEED + g_idx)
import time
from tqdm.auto import tqdm

SEED_SCONV    = 42
MAX_S_SCONV   = 50
S_VALUES      = [5, 10, 15, 20, 25, 30, 35, 40, 45, 50]
N_CONV_GRAPHS = 10

print('Loading T8 for S-Convergence...')
model_conv = load_model(T8_WEIGHTS_LOCAL, dropout=DROPOUT_MAP[8], device=DEVICE)
print('T8 loaded.')

conv_results = []
t_conv = time.time()

for g_idx in tqdm(range(N_CONV_GRAPHS), desc='S-Convergence graphs'):
    data = TEST_DATA[g_idx]

    # Run MAX_S passes at fixed seed per graph
    torch.manual_seed(SEED_SCONV + g_idx)
    model_conv.train()
    for m in model_conv.modules():
        if isinstance(m, torch.nn.modules.batchnorm._BatchNorm):
            m.eval()

    preds_all = []
    with torch.no_grad():
        for _ in range(MAX_S_SCONV):
            out = model_conv(data.to(DEVICE))
            if isinstance(out, tuple): out = out[0]
            preds_all.append(out.squeeze().cpu().numpy())
    preds_all = np.array(preds_all)   # (MAX_S, N)

    graph_entry = {'graph_idx': g_idx, 's_results': []}
    for S in S_VALUES:
        subset  = preds_all[:S]
        sigma_s = subset.std(axis=0)    # ddof=0
        graph_entry['s_results'].append({
            'S':          S,
            'mean_sigma': float(sigma_s.mean()),
        })
    conv_results.append(graph_entry)

results_sconv = {
    'convergence_graphs': conv_results,
    'S_values':           S_VALUES,
    'MAX_S':              MAX_S_SCONV,
    'seed':               SEED_SCONV,
    'n_graphs':           N_CONV_GRAPHS,
}

print(f'\nS-Convergence (mean sigma across {N_CONV_GRAPHS} graphs):')
print(f'  {"S":>4} | {"Mean Sigma":>12}')
print('  ' + '-' * 20)
for Si, S in enumerate(S_VALUES):
    vals = [g['s_results'][Si]['mean_sigma'] for g in conv_results]
    print(f'  {S:3d}  |  {np.mean(vals):.6f}')

print(f'\nTime: {(time.time()-t_conv)/60:.1f} min')

with open(os.path.join(LOCAL_PHASE3, 's_convergence_results.json'), 'w') as f:
    json.dump(results_sconv, f, indent=2)
print('Saved: s_convergence_results.json')

del model_conv
gc.collect()
if DEVICE.type == 'cuda':
    torch.cuda.empty_cache()

In [ ]:
# -- Cell 18b: S-Convergence -- Spearman rho per S value ------------------
# Extends Cell 18 by also computing Spearman rho(sigma_S, |error_S|) per S.
# Runs fresh: 10 graphs, MAX_S=50 passes, saves s_convergence_with_rho.json.
# Keeps s_convergence_results.json (Cell 18) untouched.
import time
from tqdm.auto import tqdm
from scipy import stats as scipy_stats

SEED_SCONV_RHO    = 42
MAX_S_RHO         = 50
S_VALUES_RHO      = [5, 10, 15, 20, 25, 30, 35, 40, 45, 50]
N_CONV_GRAPHS_RHO = 10

print('Loading T8 for S-Convergence with rho...')
model_rho = load_model(T8_WEIGHTS_LOCAL, dropout=DROPOUT_MAP[8], device=DEVICE)
print('T8 loaded.')

rho_results = []
t_rho = time.time()

for g_idx in tqdm(range(N_CONV_GRAPHS_RHO), desc='S-Conv+rho graphs'):
    data = TEST_DATA[g_idx]
    y_np = data.y.squeeze().cpu().numpy()   # (N,) targets

    # Run MAX_S passes at fixed seed per graph
    torch.manual_seed(SEED_SCONV_RHO + g_idx)
    model_rho.train()
    for m in model_rho.modules():
        if isinstance(m, torch.nn.modules.batchnorm._BatchNorm):
            m.eval()

    preds_all = []
    with torch.no_grad():
        for _ in range(MAX_S_RHO):
            out = model_rho(data.to(DEVICE))
            if isinstance(out, tuple): out = out[0]
            preds_all.append(out.squeeze().cpu().numpy())
    preds_all = np.array(preds_all)   # (MAX_S, N)

    graph_entry = {'graph_idx': g_idx, 's_results': []}
    for S in S_VALUES_RHO:
        subset    = preds_all[:S]
        sigma_s   = subset.std(axis=0)        # (N,) uncertainty estimate
        mu_s      = subset.mean(axis=0)       # (N,) prediction
        abs_err_s = np.abs(y_np - mu_s)       # (N,) absolute error
        rho_s, pval_s = scipy_stats.spearmanr(sigma_s, abs_err_s)
        graph_entry['s_results'].append({
            'S':             S,
            'mean_sigma':    float(sigma_s.mean()),
            'spearman_rho':  float(rho_s),
            'spearman_pval': float(pval_s),
        })
    rho_results.append(graph_entry)

# Summary table
print(f'\nS-Convergence with rho ({N_CONV_GRAPHS_RHO} graphs):')
print(f'  {"S":>4} | {"Mean Sigma":>12} | {"Mean rho":>10}')
print('  ' + '-' * 32)
for Si, S in enumerate(S_VALUES_RHO):
    sigmas = [g['s_results'][Si]['mean_sigma'] for g in rho_results]
    rhos   = [g['s_results'][Si]['spearman_rho'] for g in rho_results]
    print(f'  {S:3d}  |  {np.mean(sigmas):.6f}  |  {np.mean(rhos):.4f}')

print(f'\nTime: {(time.time()-t_rho)/60:.1f} min')

results_sconv_rho = {
    'convergence_graphs': rho_results,
    'S_values':           S_VALUES_RHO,
    'MAX_S':              MAX_S_RHO,
    'seed':               SEED_SCONV_RHO,
    'n_graphs':           N_CONV_GRAPHS_RHO,
    'note': 'spearman_rho = corr(sigma_S, |y - mu_S|) per graph per S',
}

with open(os.path.join(LOCAL_PHASE3, 's_convergence_with_rho.json'), 'w') as f:
    json.dump(results_sconv_rho, f, indent=2)
print('Saved: s_convergence_with_rho.json')

del model_rho
gc.collect()
if DEVICE.type == 'cuda':
    torch.cuda.empty_cache()


In [ ]:
# â”€â”€ Cell 19: Experiment A â€” 5 seeds, MC Dropout on T8 â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
# Seeds: [42, 142, 242, 342, 442]  (seed = 42 + run_index*100)
# Expected: MC_rho~0.4908, EnsVar_rho~0.4370, Combined_rho~0.4909
# NOTE: Set SKIP_EXP_A=False at Cell 3 to run this (takes ~5-10 hours).
from scipy.stats import spearmanr as _sp
import time
from tqdm.auto import tqdm

EXP_A_OUT = os.path.join(LOCAL_PHASE3, 'experiment_a_results.json')

if SKIP_EXP_A:
    print('SKIP_EXP_A=True â€” attempting to copy from Drive...')
    exp_a_drive = os.path.join(
        DRIVE_T8_FOLDER, 'uq_results', 'ensemble_experiments',
        'experiment_a_fixed_results.json')
    if os.path.exists(exp_a_drive):
        shutil.copy2(exp_a_drive, EXP_A_OUT)
        print(f'Copied: {exp_a_drive}')
        with open(EXP_A_OUT) as f:
            d = json.load(f)
        ens = d.get('ensemble', {})
        print(f'  From Drive: {json.dumps(ens, indent=4)}')
    elif os.path.exists(EXP_A_OUT):
        print(f'Using existing local file: {EXP_A_OUT}')
    else:
        print('WARNING: Experiment A results not found anywhere.')
        print('Set SKIP_EXP_A=False to regenerate (~5-10 hours).')

else:
    SEEDS_A = [42, 142, 242, 342, 442]
    S_A     = 30

    npz_base = np.load(os.path.join(LOCAL_RESULTS, 'mc_dropout_full_100graphs_mc30.npz'))
    y_all    = npz_base['targets'].astype(np.float64)

    all_run_preds = []
    all_run_uncs  = []

    for run_idx, seed in enumerate(SEEDS_A):
        print(f'\nExp A Run {run_idx+1}/5 (seed={seed})...')
        torch.manual_seed(seed)
        np.random.seed(seed)

        model_a = load_model(T8_WEIGHTS_LOCAL, dropout=DROPOUT_MAP[8], device=DEVICE)
        model_a.train()
        for m in model_a.modules():
            if isinstance(m, torch.nn.modules.batchnorm._BatchNorm):
                m.eval()

        run_preds, run_uncs = [], []
        for g_idx, data in enumerate(tqdm(TEST_DATA, desc=f'Run {run_idx+1}/5')):
            torch.manual_seed(seed + g_idx)
            ps = []
            with torch.no_grad():
                for _ in range(S_A):
                    out = model_a(data.to(DEVICE))
                    if isinstance(out, tuple): out = out[0]
                    ps.append(out.squeeze().cpu().numpy())
            ps = np.array(ps)
            run_preds.append(ps.mean(axis=0))
            run_uncs.append(ps.std(axis=0))   # ddof=0

        all_run_preds.append(np.concatenate(run_preds))
        all_run_uncs.append(np.concatenate(run_uncs))

        del model_a; gc.collect()
        if DEVICE.type == 'cuda': torch.cuda.empty_cache()

    all_run_preds = np.array(all_run_preds)   # (5, N)
    all_run_uncs  = np.array(all_run_uncs)    # (5, N)

    ensemble_pred = all_run_preds.mean(axis=0)
    ensemble_var  = all_run_preds.var(axis=0)           # biased by default
    mc_avg_unc    = all_run_uncs.mean(axis=0)
    combined_unc  = np.sqrt(ensemble_var + mc_avg_unc**2)

    abs_err      = np.abs(y_all - ensemble_pred)
    mc_rho,   _  = _sp(mc_avg_unc,            abs_err)
    ens_rho,  _  = _sp(np.sqrt(ensemble_var), abs_err)
    comb_rho, _  = _sp(combined_unc,          abs_err)

    results_a = {
        'MC_rho':       float(mc_rho),
        'EnsVar_rho':   float(ens_rho),
        'Combined_rho': float(comb_rho),
        'seeds':        SEEDS_A,
        'S':            S_A,
    }

    print(f'\nMC_rho:       {mc_rho:.4f}  (expected ~0.4908)')
    print(f'EnsVar_rho:   {ens_rho:.4f}  (expected ~0.4370)')
    print(f'Combined_rho: {comb_rho:.4f}  (expected ~0.4909)')

    with open(EXP_A_OUT, 'w') as f:
        json.dump(results_a, f, indent=2)
    print(f'Saved: {EXP_A_OUT}')

In [ ]:
# â”€â”€ Cell 20: Experiment B â€” R2-weighted 5-model ensemble â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
# Models: T2, T5, T6, T7, T8 deterministic
# Expected: ensemble R2~0.5656, Spearman rho~0.4333
from sklearn.metrics import r2_score as _r2
from scipy.stats import spearmanr as _sp
from tqdm.auto import tqdm
import time

EXP_B_OUT = os.path.join(LOCAL_PHASE3, 'experiment_b_results.json')

if SKIP_EXP_B:
    print('SKIP_EXP_B=True â€” attempting to copy from Drive...')
    exp_b_drive = os.path.join(
        DRIVE_T8_FOLDER, 'uq_results', 'ensemble_experiments',
        'experiment_b_fixed_results.json')
    if os.path.exists(exp_b_drive):
        shutil.copy2(exp_b_drive, EXP_B_OUT)
        print(f'Copied: {exp_b_drive}')
        with open(EXP_B_OUT) as f:
            d = json.load(f)
        ens = d.get('ensemble', {})
        print(f'  From Drive: {json.dumps(ens, indent=4)}')
    elif os.path.exists(EXP_B_OUT):
        print(f'Using existing local file: {EXP_B_OUT}')
    else:
        print('WARNING: Experiment B results not found. Set SKIP_EXP_B=False to regenerate.')

else:
    MODEL_NUMS_B = [2, 5, 6, 7, 8]
    npz_base     = np.load(os.path.join(LOCAL_RESULTS, 'mc_dropout_full_100graphs_mc30.npz'))
    y_all        = npz_base['targets'].astype(np.float64)

    all_preds_b = {}
    t_expb      = time.time()

    for m_num in MODEL_NUMS_B:
        local_w = os.path.join(LOCAL_MODELS, f't{m_num}', 'model.pth')
        if not os.path.exists(local_w):
            print(f'T{m_num}: not in local cache, trying to copy...')
            result = copy_model(m_num)
            if result is None:
                print(f'SKIP T{m_num}: weights unavailable.')
                continue

        print(f'Running T{m_num} (deterministic)...')
        model_b = load_model(local_w, dropout=DROPOUT_MAP[m_num], device=DEVICE)
        model_b.eval()

        preds_b = []
        with torch.no_grad():
            for data in tqdm(TEST_DATA, desc=f'T{m_num}', leave=False):
                out = model_b(data.to(DEVICE))
                if isinstance(out, tuple): out = out[0]
                preds_b.append(out.squeeze().cpu().numpy())
        all_preds_b[m_num] = np.concatenate(preds_b).astype(np.float64)

        r2_m  = float(_r2(y_all, all_preds_b[m_num]))
        mae_m = float(np.mean(np.abs(all_preds_b[m_num] - y_all)))
        print(f'  T{m_num}: R2={r2_m:.4f}, MAE={mae_m:.4f}')
        if r2_m < 0.3:
            print(f'  !! WARNING: T{m_num} R2 too low â€” weight loading may have an issue !!')

        del model_b; gc.collect()
        if DEVICE.type == 'cuda': torch.cuda.empty_cache()

    avail  = sorted(all_preds_b.keys())
    P      = np.array([all_preds_b[m] for m in avail])   # (K, N)
    w_vals = np.array([MODEL_WEIGHTS_R2[m] for m in avail])
    w      = w_vals / w_vals.sum()

    w_pred = np.average(P, axis=0, weights=w)
    w_var  = np.average((P - w_pred)**2, axis=0, weights=w)
    unc_b  = np.sqrt(w_var)

    r2_ens  = float(_r2(y_all, w_pred))
    mae_ens = float(np.mean(np.abs(w_pred - y_all)))
    rho_b,_ = _sp(unc_b, np.abs(w_pred - y_all))

    results_b = {
        'ensemble': {
            'r2':           r2_ens,
            'mae':          mae_ens,
            'spearman_rho': float(rho_b),
        },
        'models_used':  avail,
        'weights':      w.tolist(),
        'weights_r2':   {m: MODEL_WEIGHTS_R2[m] for m in avail},
    }

    print(f'\nExp B Ensemble R2:    {r2_ens:.4f}  (expected ~0.5656)')
    print(f'Exp B Spearman rho:   {rho_b:.4f}  (expected ~0.4333)')
    print(f'Time: {(time.time()-t_expb)/60:.1f} min')

    with open(EXP_B_OUT, 'w') as f:
        json.dump(results_b, f, indent=2)
    print(f'Saved: {EXP_B_OUT}')

In [ ]:
# â”€â”€ Cell 21: Save all phase3 results to Drive â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
DRIVE_PHASE3 = os.path.join(DRIVE_T8_FOLDER, 'uq_results', 'phase3_results')
os.makedirs(DRIVE_PHASE3, exist_ok=True)

PHASE3_FILES = [
    'crps_t8.json',
    'pit_t8.json',
    'temperature_scaling_t8.json',
    'selective_prediction_s30.json',
    'nll_results.json',
    'bootstrap_ci_results.json',
    'auroc_results.json',
    'adaptive_conformal_results.json',
    's_convergence_results.json',
    's_convergence_with_rho.json',
    'experiment_a_results.json',
    'experiment_b_results.json',
]

print('Syncing phase3 results to Drive...')
ok, skip = 0, 0
for fname in PHASE3_FILES:
    src = os.path.join(LOCAL_PHASE3, fname)
    dst = os.path.join(DRIVE_PHASE3, fname)
    if os.path.exists(src):
        shutil.copy2(src, dst)
        print(f'  OK:   {fname}')
        ok += 1
    else:
        print(f'  SKIP: {fname}  (not generated yet)')
        skip += 1

print(f'\nSynced {ok}/{len(PHASE3_FILES)} files ({skip} skipped).')
print(f'Drive location: {DRIVE_PHASE3}')

In [ ]:
# â”€â”€ Cell 22: Final Verification Table â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
# Compares computed values against thesis expected values.
# OK = within tolerance; DIFF = outside tolerance (may still be correct if thesis
#      values are rounded).

def _load(fname):
    p = os.path.join(LOCAL_PHASE3, fname)
    if not os.path.exists(p):
        return None
    with open(p) as f:
        return json.load(f)

def chk(label, computed, expected, tol=5.0):
    """tol = max acceptable % deviation from expected."""
    denom = abs(expected) if expected != 0 else 1.0
    pct   = abs(computed - expected) / denom * 100
    tag   = 'OK  ' if pct < tol else 'DIFF'
    print(f'  {tag}  {label:<40} computed={computed:>10.4f}  expected={expected:>10.4f}')

W = 80
print('=' * W)
print('  VERIFICATION TABLE â€” Computed vs Thesis Expected')
print('=' * W)

# â”€â”€ CRPS â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
d = _load('crps_t8.json')
if d:
    print('\n  [CRPS]')
    chk('CRPS mean',         d['mean_crps'],      3.38)
    chk('CRPS / MAE ratio',  d['crps_mae_ratio'], 0.857)

# â”€â”€ PIT â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
d = _load('pit_t8.json')
if d:
    print('\n  [PIT]')
    chk('PIT mean',          d['pit_mean'],       0.433)
    chk('PIT std',           d['pit_std'],        0.399)
    chk('First bin %',       d['first_bin_pct'],  28.4)
    chk('KS statistic',      d['ks_statistic'],   0.245)

# â”€â”€ Temperature Scaling â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
d = _load('temperature_scaling_t8.json')
if d:
    print('\n  [Temperature Scaling]')
    chk('T_opt',             d['T_opt'],           2.7025, tol=20.0)
    chk('ECE full',          d['ECE_full'],         0.265)
    chk('ECE test before',   d['ECE_test_before'],  0.269)
    chk('ECE test after',    d['ECE_test_after'],   0.048, tol=50.0)
    chk('ECE improvement %', d['improvement_pct'],  82.0,  tol=20.0)

# â”€â”€ NLL â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
d = _load('nll_results.json')
if d:
    print('\n  [NLL]')
    chk('NLL raw',           d['nll_raw'],  21.65)

# â”€â”€ Bootstrap CI â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
d = _load('bootstrap_ci_results.json')
if d:
    print('\n  [Bootstrap CI]')
    chk('Mean Spearman rho', d['mean_rho'], 0.464)
    chk('CI lower bound',    d['ci_lo'],    0.460)
    chk('CI upper bound',    d['ci_hi'],    0.469)

# â”€â”€ AUROC â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
d = _load('auroc_results.json')
if d:
    print('\n  [AUROC]')
    chk('AUROC',             d['auroc'],    0.755)

# â”€â”€ Selective Prediction â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
d = _load('selective_prediction_s30.json')
if d:
    print('\n  [Selective Prediction â€” MAE reduction %]')
    exp_map = {0.90: -18.3, 0.50: -41.2, 0.25: -54.6, 0.10: -73.3}
    for e in d['retention_results']:
        r = e['retention']
        if r in exp_map:
            chk(f'Retention {int(r*100):3d}%', e['mae_reduction_pct'], exp_map[r])

# â”€â”€ Adaptive Conformal â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
d = _load('adaptive_conformal_results.json')
if d:
    print('\n  [Adaptive Conformal â€” per-graph coverage %]')
    chk('Standard min cov %', d['standard_per_graph_min_pct'], 62.9, tol=10.0)
    chk('Standard max cov %', d['standard_per_graph_max_pct'], 98.6, tol=10.0)
    chk('Adaptive min cov %', d['adaptive_per_graph_min_pct'], 90.0, tol=10.0)
    chk('Adaptive max cov %', d['adaptive_per_graph_max_pct'], 96.2, tol=10.0)

# â”€â”€ Experiment A â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
d = _load('experiment_a_results.json')
if d:
    print('\n  [Experiment A]')
    # Keys may be nested (mc_dropout/ensemble_variance/combined) or flat (MC_rho etc.)
    mc_rho_v   = d.get('MC_rho',       (d.get('mc_dropout',        {}) or {}).get('spearman_rho', 0))
    ens_rho_v  = d.get('EnsVar_rho',   (d.get('ensemble_variance', {}) or {}).get('spearman_rho', 0))
    comb_rho_v = d.get('Combined_rho', (d.get('combined',          {}) or {}).get('spearman_rho', 0))
    chk('MC rho',       mc_rho_v,   0.4908)
    chk('EnsVar rho',   ens_rho_v,  0.4370)
    chk('Combined rho', comb_rho_v, 0.4909)

# â”€â”€ Experiment B â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
d = _load('experiment_b_results.json')
if d:
    print('\n  [Experiment B]')
    ens = d.get('ensemble', d)
    chk('Ensemble R2',        ens.get('r2', 0),           0.5656)
    chk('Ensemble rho',       ens.get('spearman_rho', 0), 0.4333)

print()
print('=' * W)
print('  Note: DIFF does not always mean wrong â€” thesis values are rounded.')
print('  Small differences (< 5%) in CRPS, PIT, AUROC are expected due to')
print('  float32 vs float64 and random seed propagation differences.')
print('=' * W)